# Transformer Fine-Tuning: ruBERT & XLM-RoBERTa

Fine-tune two transformer models for Russian toxic comment classification:
1. **ruBERT** — `DeepPavlov/rubert-base-cased` (Russian-specific)
2. **XLM-RoBERTa** — `xlm-roberta-base` (multilingual)

Both models use the same training pipeline with identical hyperparameters for fair comparison.

In [1]:
import sys, os, time, json

# Must be set BEFORE any transformers import to avoid Keras 3 / TF conflict
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from src.transformers_train import (
    ToxicDataset, load_splits, compute_metrics_hf,
    build_model_and_tokenizer, get_training_args, get_predictions,
)
from src.evaluation import compute_metrics, print_report, plot_confusion_matrix, plot_roc_curve, save_metrics

sns.set_theme(style="whitegrid")

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
PyTorch: 2.8.0


## 1. Load Data

In [2]:
splits = load_splits("../data")
train_texts, train_labels = splits["train"]
val_texts, val_labels = splits["val"]
test_texts, test_labels = splits["test"]

print(f"Train: {len(train_texts)}  |  Val: {len(val_texts)}  |  Test: {len(test_texts)}")

Train: 10088  |  Val: 2162  |  Test: 2162


## 2. Hyperparameters

Shared across both models for fair comparison.

In [3]:
MAX_LENGTH = 128
NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5

print(f"max_length={MAX_LENGTH}, epochs={NUM_EPOCHS}, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}")

max_length=128, epochs=3, batch_size=8, lr=2e-05


---
## 3. ruBERT — `DeepPavlov/rubert-base-cased`

In [4]:
RUBERT_NAME = "DeepPavlov/rubert-base-cased"

rubert_model, rubert_tokenizer = build_model_and_tokenizer(RUBERT_NAME)
print(f"Model parameters: {rubert_model.num_parameters():,}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model parameters: 177,854,978


In [5]:
# Tokenize datasets
train_dataset_rb = ToxicDataset(train_texts, train_labels, rubert_tokenizer, MAX_LENGTH)
val_dataset_rb = ToxicDataset(val_texts, val_labels, rubert_tokenizer, MAX_LENGTH)
test_dataset_rb = ToxicDataset(test_texts, test_labels, rubert_tokenizer, MAX_LENGTH)

print(f"Train samples: {len(train_dataset_rb)}")
print(f"Val samples:   {len(val_dataset_rb)}")
print(f"Test samples:  {len(test_dataset_rb)}")

Train samples: 10088
Val samples:   2162
Test samples:  2162


In [6]:
from transformers import Trainer

rubert_args = get_training_args(
    output_dir="../models/rubert",
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
)

rubert_trainer = Trainer(
    model=rubert_model,
    args=rubert_args,
    train_dataset=train_dataset_rb,
    eval_dataset=val_dataset_rb,
    compute_metrics=compute_metrics_hf,
)

start = time.time()
rubert_trainer.train()
train_time_rb = time.time() - start
print(f"\nruBERT training completed in {train_time_rb/60:.1f} minutes")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

### ruBERT — Evaluation

In [ ]:
# Validation
rb_val_preds, rb_val_probs, rb_val_labels = get_predictions(rubert_trainer, val_dataset_rb)
rb_val_metrics = compute_metrics(rb_val_labels, rb_val_preds, rb_val_probs)

print("--- ruBERT: Validation Set ---")
print_report(rb_val_labels, rb_val_preds)
print(f"ROC-AUC: {rb_val_metrics['roc_auc']:.4f}")

# Test
rb_test_preds, rb_test_probs, rb_test_labels = get_predictions(rubert_trainer, test_dataset_rb)
rb_test_metrics = compute_metrics(rb_test_labels, rb_test_preds, rb_test_probs)

print("\n--- ruBERT: Test Set ---")
print_report(rb_test_labels, rb_test_preds)
print(f"ROC-AUC: {rb_test_metrics['roc_auc']:.4f}")

### ruBERT — Training Loss Curve

In [ ]:
rb_log = pd.DataFrame(rubert_trainer.state.log_history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
train_log = rb_log.dropna(subset=["loss"])
axes[0].plot(train_log["step"], train_log["loss"], label="Train loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("ruBERT — Training Loss")
axes[0].legend()

# Eval metrics per epoch
eval_log = rb_log.dropna(subset=["eval_loss"])
axes[1].plot(eval_log["epoch"], eval_log["eval_f1_macro"], "o-", label="F1 macro")
axes[1].plot(eval_log["epoch"], eval_log["eval_accuracy"], "s-", label="Accuracy")
axes[1].plot(eval_log["epoch"], eval_log["eval_roc_auc"], "^-", label="ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_title("ruBERT — Validation Metrics")
axes[1].legend()
axes[1].set_ylim(0.7, 1.0)

plt.tight_layout()
plt.show()

---
## 4. XLM-RoBERTa — `xlm-roberta-base`

In [ ]:
XLM_NAME = "xlm-roberta-base"

xlm_model, xlm_tokenizer = build_model_and_tokenizer(XLM_NAME)
print(f"Model parameters: {xlm_model.num_parameters():,}")

In [ ]:
# Tokenize datasets with XLM tokenizer
train_dataset_xlm = ToxicDataset(train_texts, train_labels, xlm_tokenizer, MAX_LENGTH)
val_dataset_xlm = ToxicDataset(val_texts, val_labels, xlm_tokenizer, MAX_LENGTH)
test_dataset_xlm = ToxicDataset(test_texts, test_labels, xlm_tokenizer, MAX_LENGTH)

print("Datasets tokenized.")

In [ ]:
xlm_args = get_training_args(
    output_dir="../models/xlm-roberta",
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
)

xlm_trainer = Trainer(
    model=xlm_model,
    args=xlm_args,
    train_dataset=train_dataset_xlm,
    eval_dataset=val_dataset_xlm,
    compute_metrics=compute_metrics_hf,
)

start = time.time()
xlm_trainer.train()
train_time_xlm = time.time() - start
print(f"\nXLM-RoBERTa training completed in {train_time_xlm/60:.1f} minutes")

### XLM-RoBERTa — Evaluation

In [ ]:
# Validation
xlm_val_preds, xlm_val_probs, xlm_val_labels = get_predictions(xlm_trainer, val_dataset_xlm)
xlm_val_metrics = compute_metrics(xlm_val_labels, xlm_val_preds, xlm_val_probs)

print("--- XLM-RoBERTa: Validation Set ---")
print_report(xlm_val_labels, xlm_val_preds)
print(f"ROC-AUC: {xlm_val_metrics['roc_auc']:.4f}")

# Test
xlm_test_preds, xlm_test_probs, xlm_test_labels = get_predictions(xlm_trainer, test_dataset_xlm)
xlm_test_metrics = compute_metrics(xlm_test_labels, xlm_test_preds, xlm_test_probs)

print("\n--- XLM-RoBERTa: Test Set ---")
print_report(xlm_test_labels, xlm_test_preds)
print(f"ROC-AUC: {xlm_test_metrics['roc_auc']:.4f}")

### XLM-RoBERTa — Training Loss Curve

In [ ]:
xlm_log = pd.DataFrame(xlm_trainer.state.log_history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_log_xlm = xlm_log.dropna(subset=["loss"])
axes[0].plot(train_log_xlm["step"], train_log_xlm["loss"], label="Train loss", color="tab:orange")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("XLM-RoBERTa — Training Loss")
axes[0].legend()

eval_log_xlm = xlm_log.dropna(subset=["eval_loss"])
axes[1].plot(eval_log_xlm["epoch"], eval_log_xlm["eval_f1_macro"], "o-", label="F1 macro")
axes[1].plot(eval_log_xlm["epoch"], eval_log_xlm["eval_accuracy"], "s-", label="Accuracy")
axes[1].plot(eval_log_xlm["epoch"], eval_log_xlm["eval_roc_auc"], "^-", label="ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_title("XLM-RoBERTa — Validation Metrics")
axes[1].legend()
axes[1].set_ylim(0.7, 1.0)

plt.tight_layout()
plt.show()

---
## 5. Confusion Matrices (Test Set)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_confusion_matrix(rb_test_labels, rb_test_preds, title="ruBERT", ax=axes[0])
plot_confusion_matrix(xlm_test_labels, xlm_test_preds, title="XLM-RoBERTa", ax=axes[1])

plt.tight_layout()
plt.show()

## 6. ROC Curves (Test Set)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

plot_roc_curve(rb_test_labels, rb_test_probs, label="ruBERT", ax=ax)
plot_roc_curve(xlm_test_labels, xlm_test_probs, label="XLM-RoBERTa", ax=ax)

plt.tight_layout()
plt.show()

## 7. Results Comparison

In [ ]:
comparison = pd.DataFrame({
    "ruBERT": rb_test_metrics,
    "XLM-RoBERTa": xlm_test_metrics,
}).round(4)

comparison

## 8. Save Metrics & Models

In [ ]:
all_metrics = {
    "rubert": {
        "model_name": RUBERT_NAME,
        "val": rb_val_metrics,
        "test": rb_test_metrics,
        "train_time_minutes": round(train_time_rb / 60, 2),
        "hyperparameters": {
            "max_length": MAX_LENGTH,
            "num_epochs": NUM_EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": 0.01,
        },
    },
    "xlm_roberta": {
        "model_name": XLM_NAME,
        "val": xlm_val_metrics,
        "test": xlm_test_metrics,
        "train_time_minutes": round(train_time_xlm / 60, 2),
        "hyperparameters": {
            "max_length": MAX_LENGTH,
            "num_epochs": NUM_EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": 0.01,
        },
    },
}

save_metrics(all_metrics, "../results/transformer_metrics.json")

# Save test predictions for error analysis (Day 5)
test_df = pd.read_csv("../data/test.csv")
test_df["rubert_pred"] = rb_test_preds
test_df["rubert_prob"] = rb_test_probs
test_df["xlm_pred"] = xlm_test_preds
test_df["xlm_prob"] = xlm_test_probs
test_df.to_csv("../results/test_predictions.csv", index=False)
print("Test predictions saved to results/test_predictions.csv")

# Save best models
rubert_trainer.save_model("../models/rubert-best")
rubert_tokenizer.save_pretrained("../models/rubert-best")
xlm_trainer.save_model("../models/xlm-roberta-best")
xlm_tokenizer.save_pretrained("../models/xlm-roberta-best")
print("Models saved.")